# pyamapping - mapping functions for audio computing (and beyond)

by Thomas Hermann and Dennis Reinsch, 2023++

## Introduction 

### Background / History

- pyamapping bundles frequently used mapping functions for audio computing
- the earliest functions were reimplementations of ampdb, dbamp, midicps, cpsmidi, linlin, coded in analogy to SuperCollider3 functions to be used within the sc3nb package. (coded by TH)
- later, when I started pya, the same functions were needed, yet importing sc3nb would have caused unwanted dependencies, so we created pyamapping as a very lean package that both sc3nb and pya depend on (created by DR)
- now in 2025 pyamapping grows strongly (additions by TH) 
  - firstly by adding many mapping functions available in sc3 which were beforehand not copied
  - secondly, by introducing ChainableArray, a class that wraps numpy.ndarrays, allowing to daisy chain operations on numpy arrays, similar to how we offer it for pya.
- This notebook introduces the available mapping functions with examples.

### Overview

**pyamapping** offers a set of mapping functions often used 

- in the context of sound and computer music 
- in the context of auditory display and sonification (e.g. parameter mapping sonifications)
- ...

A source of inspiration is librosa and Supercollider3. This package reimplements them and adds mappings used in the interactive sonification stack (cf. <https://github.com/interactive-sonification>), including the following packages that all make use of pyamapping:

- **sc3nb** - sc3 interface for Python and Jupyter notebooks
- **pya**  - the python Audio Coding Package
- **mesonic** - a middleware for sonification and auditory display 
- **sonecules** - a high-level class library for sonification and auditory display

**Chainable numpy arrays**

Method chaining offers concise syntax and proved helpful in pya.
Numpy offers method chaining only for few functions.
This package extends method chaining 

- by inheriting the class `ChainableArray` from `numpy.ndarray`
- by adding wrappers to enable a method chain syntax for most `ufuncs`
- by providing a general `map()` method for direct vectorized mapping
- by providing helper functions to vectorize Python functions into methods

Furthermore it offers some convenience functions, e.g.

- to plot arrays (optionally as signal with given sample rate)

We hope that pyamapping will help to write signal transformations and manipulations in a more concise, compact and readible manner.

**Imports and Headers**

- as pyamapping is a long name, importing as `pam` is a suggested abbreviation 
- matplotlib and pprint imports are merely for showing example output

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from pprint import pprint
import pyamapping as pam

mpl.rcParams['figure.figsize'] = (9,2.5)

## Available pyamapping functions - Overview

- in import of pyamapping, wrappers are automatically created for numpy ufuncs.
  - later versions may have them created verbatim to enable code completion
- the following code simply lists those numpy functions plus special dedicated/ new pyamapping functions that do not have their origin in numpy

In [ ]:
from pyamapping.mappings import _list_numpy_ufuncs, pyamapping_functions

# print (i) a compact list of all unary and binary numpy functions 
# and (ii) all pyamapping functions
u_lists = [[], []]

for ufunc in _list_numpy_ufuncs():
    u_lists[ufunc.nin - 1].append(ufunc.__name__)

# compact list numpy functions
for i, li in enumerate(u_lists):
    print(f"\n=== numpy functions with {i+1} argument ===")
    pprint(li, compact=True, width=80)

# compact list of pyamapping functions
li = [el.__name__ for el in pyamapping_functions]
print(f"\n=== pyamapping functions: ===")
pprint(li, compact=True, width=80)

## pyamapping - Demonstration and Examples

### ChainableArray - Basics

Any numpy array can be turned into a chainable array by using the `ChainableArray` class defined in `pyamapping`.
- the chain() function provides a shortcut, making this construction shorter.


In [ ]:
from pyamapping import ChainableArray, chain

# some data
data = np.random.random(100)

# create ChainableArray
dch = ChainableArray(data)

# the same can be obtained shorter by
dch = chain(data)

ChainableArray offer the following methods:

- `to_array` - back to numpy ndarray
- `to_asig` - convert into pya.Asig
- `plot` - plot signal(s) as time series
- `mapvec` - map function on self by using numpy.vectorize
- `map` - apply function directly to the array itself

Here is a quick demonstration:

- let us
  - map $x \to (5x)^2 + 0.1$, 
  - plot as signal assuming sampling rate 100 Hz, 
  - convert into decibel, 
  - turn that into an audio signal (i.e. pya.Asig) 
  - and plot it in the same figure created above.
  - finally transform the ChainableArray back to a regular numpy.ndarray.

Using pyamapping, the code is both shorter and more concise than the above description:

In [ ]:
dch2 = dch.map(lambda x: (5*x)**2+0.1).plot(sr=100, color="r").mapvec(pam.amp_to_db)
a1 = dch2.to_asig(sr=100).plot(color="c", lw=0.8)

# conversion back to numpy array is rarely needed but if...
dd = dch2.to_array()
type(dd)

ChainableArray is a recent addition to pyamapping, yet introduced here as it makes demonstrations of mapping functions extremely readable...